<a href="https://colab.research.google.com/github/Sevi-11/UnsupervisedLearning-Apriori/blob/main/Apriori_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module='jupyter_client')

In [2]:
# ==========================================
# 1. THE DATA PREPARATION
# ==========================================
# We use a list of lists to represent transactions.
# This matches the "Toy Dataset" from the manual walkthrough.
dataset = [
    ['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
    ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
    ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
    ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
    ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice Cream', 'Eggs']
]

# The Apriori algorithm requires a One-Hot Encoded DataFrame.
# (True if item is present, False if not).
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("=== The One-Hot Encoded Data ===")
print(df.head())
print("\n")

=== The One-Hot Encoded Data ===
   Apple   Corn   Dill   Eggs  Ice Cream  Kidney Beans   Milk  Nutmeg  Onion  \
0  False  False  False   True      False          True   True    True   True   
1  False  False   True   True      False          True  False    True   True   
2   True  False  False   True      False          True   True   False  False   
3  False   True  False  False      False          True   True   False  False   
4  False   True  False   True       True          True  False   False   True   

   Unicorn  Yogurt  
0    False    True  
1    False    True  
2    False   False  
3     True    True  
4    False   False  




In [3]:
# ==========================================
# 2. GENERATE FREQUENT ITEMSETS (The Pruning)
# ==========================================
# We set min_support to 0.6 (60%) as per the walkthrough.
# use_colnames=True allows us to see names like 'Milk' instead of column indices.

frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)

# Add a length column to help us see the "Level" (1-itemset, 2-itemset, etc.)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("=== Frequent Itemsets (After Pruning) ===")
# Sorting by Support to see the most popular items first
print(frequent_itemsets.sort_values(by='support', ascending=False))
print("\n")


=== Frequent Itemsets (After Pruning) ===
    support                     itemsets  length
1       1.0               (Kidney Beans)       1
0       0.8                       (Eggs)       1
5       0.8         (Kidney Beans, Eggs)       2
2       0.6                       (Milk)       1
3       0.6                      (Onion)       1
4       0.6                     (Yogurt)       1
6       0.6                (Onion, Eggs)       2
7       0.6         (Milk, Kidney Beans)       2
8       0.6        (Kidney Beans, Onion)       2
9       0.6       (Kidney Beans, Yogurt)       2
10      0.6  (Kidney Beans, Onion, Eggs)       3




In [4]:
# ==========================================
# 3. GENERATE RULES (The Logic)
# ==========================================
# We generate rules from the frequent itemsets.
# We set min_threshold (Confidence) to 0.7 (70%).
# We also calculate 'lift' and 'conviction'.

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# We filter the columns to show only the most relevant metrics
output_columns = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
final_rules = rules[output_columns].sort_values(by='lift', ascending=False)

print("=== Final Association Rules ===")
print(final_rules)

=== Final Association Rules ===
              antecedents            consequents  support  confidence  lift
2                 (Onion)                 (Eggs)      0.6        1.00  1.25
7   (Kidney Beans, Onion)                 (Eggs)      0.6        1.00  1.25
10                (Onion)   (Kidney Beans, Eggs)      0.6        1.00  1.25
3                  (Eggs)                (Onion)      0.6        0.75  1.25
8    (Kidney Beans, Eggs)                (Onion)      0.6        0.75  1.25
11                 (Eggs)  (Kidney Beans, Onion)      0.6        0.75  1.25
0          (Kidney Beans)                 (Eggs)      0.8        0.80  1.00
1                  (Eggs)         (Kidney Beans)      0.8        1.00  1.00
6                (Yogurt)         (Kidney Beans)      0.6        1.00  1.00
5                 (Onion)         (Kidney Beans)      0.6        1.00  1.00
4                  (Milk)         (Kidney Beans)      0.6        1.00  1.00
9           (Onion, Eggs)         (Kidney Beans)      0.

/usr/local/lib/python3.12/dist-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [5]:
# ==========================================
# 4. ENGINEERING INTERPRETATION
# ==========================================
print("\n=== Interpretation of the Top Rule ===")
if not final_rules.empty:
    top_rule = final_rules.iloc[0]
    ant = list(top_rule['antecedents'])[0]
    con = list(top_rule['consequents'])[0]
    lift = top_rule['lift']

    print(f"Rule: If a user buys {ant}, they also buy {con}.")
    print(f"Lift: {lift:.2f}")
    if lift > 1:
        print("Conclusion: Positive Correlation (Good Rule)")
    elif lift < 1:
        print("Conclusion: Negative Correlation (Substitute items)")
    else:
        print("Conclusion: No Correlation (Coincidence)")
else:
    print("No rules met the criteria.")


=== Interpretation of the Top Rule ===
Rule: If a user buys Onion, they also buy Eggs.
Lift: 1.25
Conclusion: Positive Correlation (Good Rule)
